Goal: 
to normalize the energy (given by sc_integral) of each event by:
 - Iron calibration peak (given by the gaussian mean of 55Fe spectra for the selected reference position)


In [ ]:
import pandas as pd
import cygno as cygno
import uproot
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
import os
import warnings
warnings.filterwarnings('ignore')

from configs import *
from functions import *
from cuts_cygno import fiducial_cuts_flaminia_dict, create_mask

In [ ]:
# Output folder
main_folder_to_save = '../002_energy_normalisation_SAVE_DATA'
os.makedirs(main_folder_to_save, exist_ok=True)

In [ ]:
RUN  = 'Run 4'
run_min, run_max = lime_underground_run_numbers_dict[RUN]

In [ ]:
logbook_df = cygno.read_cygno_logbook()
logbook_df = logbook_df.rename(columns={'run_number': 'run'})

GEM_VOLTAGES = {'HG': 440, 
                'LG': 400
               }

base_mask = (
    (logbook_df['source_type'] == 1) &
    (logbook_df['run'] >= run_min) &
    (logbook_df['run'] <= run_max) &
    (logbook_df['run_description'].isin(steps_of_interest))
)

selected_runs = {}
for gain_label, voltage in GEM_VOLTAGES.items():
    mask = base_mask & (logbook_df['GEM1_V'] == voltage)
    df   = logbook_df[mask].copy()
    # Correct source position and add step label
    df['corrected_source_position'] = df['source_position'].map(wrong_to_wright_position_dict)
    df['step'] = df['corrected_source_position'].map(correct_position_to_step_dict)
    selected_runs[gain_label] = df
    print(f"{gain_label} ({voltage} V): {len(df)} runs found")

In [ ]:
data_directory = f"/jupyter-workspace/cnaf-storage/cygno-analysis/RECO/Run4"

jagged_data = {}
for gain_label, runs_df in selected_runs.items():
    run_list = runs_df['run'].unique()
    print(f"\nLoading {gain_label} data ({len(run_list)} runs)...")
    df = load_jagged_data(run_list, data_directory)
    df = df.rename(columns={'Run': 'run'})
    # Merge logbook info (exposure_sec, step, position, GEM voltage)
    df = df.merge(
        runs_df[['run', 'exposure_sec', 'corrected_source_position', 'step', 'GEM1_V']],
        on='run', how='left'
    )
    jagged_data[gain_label] = df
    print(f"  -> {len(df)} events loaded for {gain_label}")

In [ ]:
# applying cuts to jagged data
for gain_label in jagged_data.keys():
    df = jagged_data[gain_label].copy()
    mask = create_mask(df, fiducial_cuts_flaminia_dict)
    jagged_data[gain_label] = df[mask]

In [ ]:
rectangular_data = {}
for gain_label, runs_df in selected_runs.items():
    run_list = runs_df['run'].unique()
    print(f"\nLoading {gain_label} data ({len(run_list)} runs)...")
    df = load_rectangular_data_fast(run_list, data_directory)
 
    # Merge logbook info (exposure_sec, step, position, GEM voltage)
    df = df.merge(
        runs_df[['run','start_time', 'exposure_sec', 'corrected_source_position', 'step', 'GEM1_V']],
        on='run', how='left'
    )
    rectangular_data[gain_label] = df
    print(f"  -> {len(df)} events loaded for {gain_label}")

In [ ]:
CALIBRATION_POSITION = 25# in cm
SC_VAR               = 'sc_integral'
DAYS_INTERVAL = 5 # number of days for which to match an High gain and a Low gain run

FIT_LIMITS = {
    'HG': {'x_min': 2500,  'x_max': 20000, 'y_max': 300, 'n_bins': 80},
    'LG': {'x_min': 600,  'x_max': 2800, 'y_max': 300, 'n_bins': 80},
}
fe55_peak = {}   # will store the calibration peak mean for each gain

In [ ]:
# merging HG and LG runs in corresponding histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (gain_label, df) in zip(axes, jagged_data.items()):
    lims = FIT_LIMITS[gain_label]

    # Select only step-5 events
    cal_data = df[
        (df['corrected_source_position'] == CALIBRATION_POSITION) &
        (df[SC_VAR] > lims['x_min']) &
        (df[SC_VAR] < lims['x_max'])
    ][SC_VAR]

    print(f"{gain_label}: {len(cal_data)} events at {CALIBRATION_POSITION} cm for calibration fit")

    hist, bin_edges = np.histogram(cal_data, bins=lims['n_bins'])
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width   = bin_edges[1] - bin_edges[0]

    def gaussian(x, A, mu, sigma):
        return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

    from scipy.optimize import curve_fit
    p0 = [hist.max(), bin_centers[hist.argmax()], (bin_edges[-1]-bin_edges[0])/6]
    params, cov = curve_fit(gaussian, bin_centers, hist, p0=p0,
                            bounds=([0, -np.inf, 0], [np.inf, np.inf, np.inf]))
    A_fit, mu_fit, sigma_fit = params
    errs = np.sqrt(np.diag(cov))

    fe55_peak[gain_label] = mu_fit
    print(f"  Fe-55 peak ({gain_label}): mu = {mu_fit:.1f} ± {errs[1]:.1f}")

    # Plot
    ax.bar(bin_centers, hist, width=bin_width, alpha=0.5, label=f'Data ({CALIBRATION_POSITION} cm)')
    x_fit = np.linspace(bin_edges[0], bin_edges[-1], 500)
    ax.plot(x_fit, gaussian(x_fit, *params), 'r-', label=f'Gaussian fit\nμ={mu_fit:.1f}, σ={sigma_fit:.1f}')
    ax.axvline(mu_fit, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel(SC_VAR)
    ax.set_ylabel('Counts')
    ax.set_title(f'Fe-55 calibration — {gain_label} ({GEM_VOLTAGES[gain_label]} V)- All data | no correct')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(main_folder_to_save, 'fe55_calibration_HG_LG.png'), dpi=150)
plt.show()

print(f"\nCalibration peaks:")
for g, peak in fe55_peak.items():
    print(f"  {g}: {peak:.1f} ADC counts")

In [ ]:
lg_df = pd.DataFrame({'LG_start_time': selected_runs['LG'].start_time.sort_values().values})
hg_df = pd.DataFrame({'HG_start_time': selected_runs['HG'].start_time.sort_values().values})

matched_df = merge_asof(
    lg_df, hg_df,
    left_on='LG_start_time',
    right_on='HG_start_time',
    tolerance=pd.Timedelta(days=DAYS_INTERVAL),
    direction='nearest'
)
print(matched_df)

In [ ]:
if selected_runs['HG'].exposure_sec.unique() == selected_runs['LG'].exposure_sec.unique():
    print(f"HG and LG have same exposure time")

In [ ]:
from pandas import merge_asof

step = correct_position_to_step_dict[CALIBRATION_POSITION]
lg_df = (selected_runs['LG'][selected_runs['LG'].step == step]
         [['start_time', 'run']]
         .sort_values('start_time')
         .rename(columns={'start_time': 'LG_start_time', 'run': 'LG_run'}))

hg_df = (selected_runs['HG'][selected_runs['HG'].step == step]
         [['start_time', 'run']]
         .sort_values('start_time')
         .rename(columns={'start_time': 'HG_start_time', 'run': 'HG_run'}))

matched_df = merge_asof(
    lg_df, hg_df,
    left_on ='LG_start_time',
    right_on='HG_start_time',
    tolerance=pd.Timedelta(days=DAYS_INTERVAL),
    direction='nearest'
)

for row in matched_df.iterrows():
    inst_LG_start_time = row[1]['LG_start_time']
    inst_HG_start_time = row[1]['HG_start_time']
    inst_LG_run        = row[1]['LG_run']
    inst_HG_run        = row[1]['HG_run']
    if pd.isna(inst_HG_start_time): continue
    print(f'match: LG run {int(inst_LG_run)} ({inst_LG_start_time})')
    print(f'     : HG run {int(inst_HG_run)} ({inst_HG_start_time})')

In [ ]:
sc_variable = 'sc_integral'
COLORS = {'HG': 'steelblue', 'LG': 'darkorange'}

for _, row in matched_df.iterrows():
    inst_LG_run        = (row['LG_run'])
    inst_HG_run        = (row['HG_run'])
    inst_LG_start_time = row['LG_start_time']
    inst_HG_start_time = row['HG_start_time']

    if pd.isna(inst_HG_start_time):
        continue
    inst_LG_run        = int(inst_LG_run)
    inst_HG_run        = int(inst_HG_run)
    # Load data for this pair
    data = {
        'HG': jagged_data['HG'][jagged_data['HG']['run'] == inst_HG_run][sc_variable],
        'LG': jagged_data['LG'][jagged_data['LG']['run'] == inst_LG_run][sc_variable],
    }

    fig = plt.figure(figsize=(14, 10))
    # Top row: side by side
    ax_hg  = fig.add_subplot(2, 2, 1)
    ax_lg  = fig.add_subplot(2, 2, 2)
    # Bottom: superimposed (spans both columns)
    ax_sup = fig.add_subplot(2, 1, 2)

    results = {}
    for gain_label, ax, color in zip(['HG', 'LG'], [ax_hg, ax_lg], ['steelblue', 'darkorange']):
        params, errs, resolution = fit_and_plot_histogram(ax, data[gain_label], gain_label, color, FIT_LIMITS)
        results[gain_label] = {'params': params, 'errs': errs, 'resolution': resolution}

    # Bottom plot: superimposed normalised histograms
    for gain_label, color in COLORS.items():
        lims = FIT_LIMITS[gain_label]
        d    = data[gain_label]
        d    = d[(d > lims['x_min']) & (d < lims['x_max'])]
        mu   = results[gain_label]['params'][1]
        sig  = results[gain_label]['params'][2]
        res  = results[gain_label]['resolution']

        # Normalise sc_integral to Fe-55 peak so both gains overlap
        ax_sup.hist(d / mu, bins=80, alpha=0.5, color=color, density=True,
                    label=f"{gain_label}  μ={mu:.1f}  R={res:.1f}%")

    ax_sup.axvline(1.0, color='red', linestyle='--', linewidth=1, label='Fe-55 peak')
    ax_sup.set_xlabel(f'{sc_variable} / μ_Fe55')
    ax_sup.set_ylabel('Normalised counts')
    ax_sup.set_title('HG vs LG — superimposed (normalised to Fe-55 peak)')
    ax_sup.legend()

    fig.suptitle(f'HG run {inst_HG_run} ({inst_HG_start_time.date()})  ↔  '
                 f'LG run {inst_LG_run} ({inst_LG_start_time.date()})', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(main_folder_to_save,
                             f'crosscheck_HG{inst_HG_run}_LG{inst_LG_run}.png'), dpi=150)
    plt.show()

Conclusions:
- iron calibration low gain (LG) data is sparce (only 9 calibration sets have been found for run4)
- a comparison between LG and high gain (HG) data was performed, matching HG and LG datasets taken within a 5 day period of each other 
- LG data has worst energy resolution, showing a broader spectra than HG data. Examples of the plots can be generated with the notebook "002_energy_normalization_HG_LH.ipynb".
- the energy resolution (ER) for HG is better than the one for LG by > 10% (typical values of ER are around 30% for HG,and larger than 45% for LG. only fiducial cuts have been applied for this analysis)

Notes:
- livetime on both datasets is identical, so normalization to livetime was not performed
- due to the sparcicity of the LG data, and the fact that the LG datasets are concentrated on time, most of the LG runs used in this comparison have as a matching HG run (the closest in time) the same run (HG run 50896 (2024-03-05))